# 02 — CNN Training
### Digit Recognizer — CNN Project | Notebook 2 of 3

**Environment:** Google Colab (GPU recommended: Runtime → Change runtime type → GPU)

**Scope:** Build `SimpleCNN` (existing architecture, preserved), train with
and without augmentation (existing logic, preserved), and add checkpointing,
an LR scheduler, and early stopping (Recommended Enhancements).

This notebook reloads the data independently (same `random_state=42` split
as Notebook 1) so it can run standalone.

In [1]:
import os, random, time, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## 1. Load data (same split as Notebook 1)

**Recommended Enhancement:** `torch.manual_seed` / `cuda.manual_seed_all`
added on top of the original `numpy`/sklearn seeding — the original
notebook had no torch-level seeding, so CNN weight init and augmentation
randomness were not reproducible before.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# from google.colab import drive
# drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/digit-recognizer-cnn"
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

X = train.drop(columns=['label'])
y = train['label']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)
print(X_train.shape, X_val.shape, test.shape)

(37800, 784) (4200, 784) (28000, 784)


## 2. Datasets — **Existing Implementation, preserved**

`DigitDataset` and `AugmentedDigitDataset` (rotation ±10°, shift ±2px via
affine grid sampling + roll) are unchanged from the original notebook.

In [4]:
class DigitDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = X.values.reshape(-1, 1, 28, 28).astype('float32') / 255.0
        self.y = y.values.astype('int64') if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        img = torch.tensor(self.X[idx])
        if self.y is not None:
            return img, torch.tensor(self.y[idx])
        return img


class AugmentedDigitDataset(Dataset):
    def __init__(self, X, y=None, rotate_deg=10, shift_px=2):
        self.X = X.values.reshape(-1, 1, 28, 28).astype('float32') / 255.0
        self.y = y.values.astype('int64') if y is not None else None
        self.rotate_deg = rotate_deg
        self.shift_px = shift_px

    def __len__(self):
        return len(self.X)

    def _rotate(self, img):
        angle = (random.random() * 2 - 1) * self.rotate_deg * (np.pi / 180)
        theta = torch.tensor([
            [np.cos(angle), -np.sin(angle), 0],
            [np.sin(angle), np.cos(angle), 0]
        ], dtype=torch.float32).unsqueeze(0)
        grid = F.affine_grid(theta, img.unsqueeze(0).size(), align_corners=False)
        rotated = F.grid_sample(img.unsqueeze(0), grid, align_corners=False)
        return rotated.squeeze(0)

    def __getitem__(self, idx):
        img = torch.tensor(self.X[idx])
        img = self._rotate(img)
        shift_x = random.randint(-self.shift_px, self.shift_px)
        shift_y = random.randint(-self.shift_px, self.shift_px)
        img = torch.roll(img, shifts=(shift_y, shift_x), dims=(1, 2))
        if self.y is not None:
            return img, torch.tensor(self.y[idx])
        return img


train_loader = DataLoader(DigitDataset(X_train, y_train), batch_size=128, shuffle=True)
val_loader = DataLoader(DigitDataset(X_val, y_val), batch_size=128, shuffle=False)
aug_train_loader = DataLoader(AugmentedDigitDataset(X_train, y_train), batch_size=128, shuffle=True)
print("Loaders ready.")

Loaders ready.


## 3. Model — `SimpleCNN` — **Existing Implementation, preserved exactly**

This exact class definition is duplicated in `app/model_def.py` so the
Streamlit app loads the identical architecture — the app never redefines
the model independently.

In [5]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.dropout = nn.Dropout(0.25)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

## 4. Training loop — **Enhanced**

**Existing Implementation:** Adam optimizer, CrossEntropyLoss, fixed epoch
count, per-epoch validation accuracy printout.

**Recommended Enhancements (each justified):**
- **Model checkpointing** — saves the model state with the *best* validation
  accuracy, not just whatever the last epoch happens to produce. The
  original notebook's `model2` kept epoch 15's weights even though epoch 12
  scored higher on validation — checkpointing fixes this data loss.
- **`ReduceLROnPlateau` LR scheduler** — lowers the learning rate when
  validation accuracy stalls, helping the model settle into a better
  minimum instead of oscillating near convergence.
- **Early stopping** — stops training if validation accuracy doesn't
  improve for `patience` epochs, saving compute and avoiding overfitting.
- **Training history dict** — records per-epoch train/val loss & accuracy
  so Notebook 3 can plot curves (this data wasn't captured before).

In [6]:
def train_model(model, train_loader, val_loader, epochs, lr=1e-3,
                 patience=5, ckpt_path=None, run_name="run"):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2
    )

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, lbls)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == lbls).sum().item()
            total += lbls.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, lbls)
                val_loss += loss.item() * imgs.size(0)
                val_correct += (outputs.argmax(1) == lbls).sum().item()
                val_total += lbls.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total
        scheduler.step(val_acc)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"[{run_name}] Epoch {epoch+1}/{epochs} | "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_no_improve = 0
            if ckpt_path:
                torch.save(model.state_dict(), ckpt_path)
                print(f"  -> New best ({val_acc:.4f}). Saved checkpoint to {ckpt_path}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  -> Early stopping triggered (no improvement for {patience} epochs).")
                break

    return history, best_val_acc

## 5. Run 1 — CNN without augmentation — **Existing Implementation, preserved**

Same setup as the original notebook (`Adam(lr=0.001)`, 10 epochs), now with
checkpointing/scheduler/early-stopping wrapped around it.

In [7]:
set_seed(SEED)
model = SimpleCNN().to(device)
run1_ckpt = os.path.join(MODELS_DIR, "run1_best.pth")

history_run1, best_acc_run1 = train_model(
    model, train_loader, val_loader,
    epochs=10, lr=1e-3, patience=5,
    ckpt_path=run1_ckpt, run_name="run1_no_aug"
)
print("Run 1 best val accuracy:", best_acc_run1)

[run1_no_aug] Epoch 1/10 | train_loss=0.2036 train_acc=0.9358 | val_loss=0.0646 val_acc=0.9793
  -> New best (0.9793). Saved checkpoint to /content/drive/MyDrive/digit-recognizer-cnn/models/run1_best.pth
[run1_no_aug] Epoch 2/10 | train_loss=0.0651 train_acc=0.9798 | val_loss=0.0490 val_acc=0.9855
  -> New best (0.9855). Saved checkpoint to /content/drive/MyDrive/digit-recognizer-cnn/models/run1_best.pth
[run1_no_aug] Epoch 3/10 | train_loss=0.0520 train_acc=0.9840 | val_loss=0.0419 val_acc=0.9867
  -> New best (0.9867). Saved checkpoint to /content/drive/MyDrive/digit-recognizer-cnn/models/run1_best.pth
[run1_no_aug] Epoch 4/10 | train_loss=0.0378 train_acc=0.9878 | val_loss=0.0441 val_acc=0.9855
[run1_no_aug] Epoch 5/10 | train_loss=0.0336 train_acc=0.9892 | val_loss=0.0424 val_acc=0.9867
[run1_no_aug] Epoch 6/10 | train_loss=0.0272 train_acc=0.9913 | val_loss=0.0418 val_acc=0.9881
  -> New best (0.9881). Saved checkpoint to /content/drive/MyDrive/digit-recognizer-cnn/models/run1_bes

## 6. Run 2 — CNN with augmentation — **Existing Implementation, preserved**

Same setup as the original notebook (fresh `SimpleCNN`, augmented loader,
15 epochs), now checkpointed as `best_model.pth` and the last-epoch state
also saved as `final_model.pth` for comparison.

In [8]:
set_seed(SEED)
model2 = SimpleCNN().to(device)
best_ckpt = os.path.join(MODELS_DIR, "best_model.pth")

history_run2, best_acc_run2 = train_model(
    model2, aug_train_loader, val_loader,
    epochs=15, lr=1e-3, patience=5,
    ckpt_path=best_ckpt, run_name="run2_augmented"
)
print("Run 2 best val accuracy:", best_acc_run2)

# Also persist the final-epoch state (may differ from best_model.pth)
final_ckpt = os.path.join(MODELS_DIR, "final_model.pth")
torch.save(model2.state_dict(), final_ckpt)
print("Saved final-epoch model to:", final_ckpt)

[run2_augmented] Epoch 1/15 | train_loss=0.3341 train_acc=0.8945 | val_loss=0.0609 val_acc=0.9795
  -> New best (0.9795). Saved checkpoint to /content/drive/MyDrive/digit-recognizer-cnn/models/best_model.pth
[run2_augmented] Epoch 2/15 | train_loss=0.1272 train_acc=0.9595 | val_loss=0.0477 val_acc=0.9829
  -> New best (0.9829). Saved checkpoint to /content/drive/MyDrive/digit-recognizer-cnn/models/best_model.pth
[run2_augmented] Epoch 3/15 | train_loss=0.0942 train_acc=0.9708 | val_loss=0.0399 val_acc=0.9871
  -> New best (0.9871). Saved checkpoint to /content/drive/MyDrive/digit-recognizer-cnn/models/best_model.pth
[run2_augmented] Epoch 4/15 | train_loss=0.0793 train_acc=0.9756 | val_loss=0.0440 val_acc=0.9876
  -> New best (0.9876). Saved checkpoint to /content/drive/MyDrive/digit-recognizer-cnn/models/best_model.pth
[run2_augmented] Epoch 5/15 | train_loss=0.0801 train_acc=0.9752 | val_loss=0.0370 val_acc=0.9893
  -> New best (0.9893). Saved checkpoint to /content/drive/MyDrive/dig

## 7. Save training history — **New**

Persists both runs' per-epoch metrics so Notebook 3 can plot loss/accuracy
curves without retraining.

In [9]:
training_history = {
    "run1_no_aug": history_run1,
    "run2_augmented": history_run2,
    "best_val_acc_run1": best_acc_run1,
    "best_val_acc_run2": best_acc_run2,
}

with open(os.path.join(OUTPUTS_DIR, "training_history.pkl"), "wb") as f:
    pickle.dump(training_history, f)

print("Saved:", os.path.join(OUTPUTS_DIR, "training_history.pkl"))

Saved: /content/drive/MyDrive/digit-recognizer-cnn/outputs/training_history.pkl


## 8. Summary & Next Steps

- `models/run1_best.pth` — best checkpoint from the non-augmented run (kept for comparison).
- `models/best_model.pth` — best checkpoint from the augmented run (**the model the app will use**).
- `models/final_model.pth` — last-epoch weights of the augmented run.
- `outputs/training_history.pkl` — per-epoch metrics for both runs.

**Next:** `03_Evaluation_and_Export.ipynb` — confusion matrix, classification
report, curves, and the final Kaggle `submission.csv`.